In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import pandas as pd


df = pd.read_csv('../data/processed/df_model_baseline_v1.csv')

In [2]:
features = [
    'txn_count',
    'has_txn_history',
    'auto_renew_last',
    'payment_plan_days_last',
    'plan_list_price_last',
    'actual_amount_paid_last',
    'cancel_count',
    'has_cancelled',
    'days_since_registration'
]

X = df[features]
y = df['is_churn']

In [3]:
X = X.fillna(0)

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [6]:
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]

In [7]:
print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC AUC:", roc_auc_score(y_val, y_proba))

print(classification_report(y_val, y_pred))

Accuracy: 0.9142549641591826
ROC AUC: 0.5672750044841146
              precision    recall  f1-score   support

           0       0.92      1.00      0.95    176726
           1       0.78      0.06      0.12     17466

    accuracy                           0.91    194192
   macro avg       0.85      0.53      0.54    194192
weighted avg       0.90      0.91      0.88    194192



In [8]:
coef = pd.DataFrame({
    'feature': features,
    'coef': model.coef_[0]
}).sort_values(by='coef', ascending=False)

print(coef)

                   feature      coef
1          has_txn_history  4.104412
7            has_cancelled  2.169837
3   payment_plan_days_last  0.205587
0                txn_count  0.076635
5  actual_amount_paid_last  0.003992
8  days_since_registration  0.000027
4     plan_list_price_last -0.045120
6             cancel_count -1.164460
2          auto_renew_last -3.933610


In [9]:
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    
    print(f"\n===== Threshold: {t} =====")
    print("Precision:", precision_score(y_val, y_pred_t, zero_division=0))
    print("Recall:", recall_score(y_val, y_pred_t, zero_division=0))
    print("F1:", f1_score(y_val, y_pred_t, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred_t))


===== Threshold: 0.5 =====
Precision: 0.7808407994486561
Recall: 0.06486888812550097
F1: 0.11978643548131311
Confusion matrix:
[[176408    318]
 [ 16333   1133]]

===== Threshold: 0.4 =====
Precision: 0.7016328550222662
Recall: 0.08118630482079468
F1: 0.1455329193821522
Confusion matrix:
[[176123    603]
 [ 16048   1418]]

===== Threshold: 0.3 =====
Precision: 0.676005422503389
Recall: 0.08565212412687508
F1: 0.1520402459474567
Confusion matrix:
[[176009    717]
 [ 15970   1496]]

===== Threshold: 0.2 =====
Precision: 0.6417963224893918
Recall: 0.10391618000687049
F1: 0.17887060214841824
Confusion matrix:
[[175713   1013]
 [ 15651   1815]]

===== Threshold: 0.1 =====
Precision: 0.17245332459875531
Recall: 0.18086568189625557
F1: 0.17655935613682092
Confusion matrix:
[[161567  15159]
 [ 14307   3159]]


In [10]:
results = []

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_val, y_pred_t, zero_division=0),
        'recall': recall_score(y_val, y_pred_t, zero_division=0),
        'f1': f1_score(y_val, y_pred_t, zero_division=0),
    })

threshold_results = pd.DataFrame(results)
print(threshold_results)

   threshold  precision    recall        f1
0        0.5   0.780841  0.064869  0.119786
1        0.4   0.701633  0.081186  0.145533
2        0.3   0.676005  0.085652  0.152040
3        0.2   0.641796  0.103916  0.178871
4        0.1   0.172453  0.180866  0.176559
